# Module 1 Assignment: Implementing a Multi-Agent Automatic Code Review 

Welcome to the first graded assignment of the course! 

## Background

Your development team is implementing a continuous integration pipeline and wants to automate initial code reviews in order to save valuable time. 

## General instructions for grading
- Replace all `None` instances with your own solution.
- You can add new cells to experiment, but these will be omitted by the grader. Only use the provided cells for your solution code.
- Before submitting, make sure all the cells in your lab work correctly.
- **Do not change variable names**: if you modify variable names, the grader won't be able to find your solutions
- **Use the provided configuration**: for grading, please use all provided configurations. Don't change the configuration files or settings. You can experiment after submitting your lab.
- To submit your notebook, save it and then click on the red **Submit Assignment** button at the top right of the page.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of Contents

- [1 - Understanding the context](#1)
- [2 - Set up your notebook](#2)
  - [2.1 - Import modules](#2-1)
- [3 - Define the elements for your Crew](#3)
  - [3.1 - Providing tools for your agents](#3-1)
    - [Exercise 1: Create tool instances](#ex1)
  - [3.2 - Define the agents](#3-2)
    - [Exercise 2: Senior Developer agent](#ex2)
    - [Exercise 3: Security Engineer agent](#ex3)
    - [Exercise 4: Tech Lead agent](#ex4)
  - [3.3 - Define the Tasks for each agent](#3-3)
    - [Exercise 5: Create Quality Analysis Task](#ex5)
    - [Exercise 6: Create Security Review Task](#ex6)
    - [Exercise 7: Create Review Decision Task](#ex7)
- [4 - Define and kick off your Crew](#4)
  - [Exercise 8: Define your Crew](#ex8)
  - [Exercise 9: Kickoff your Crew](#ex9)

<a id='1'></a>

## 1 - Understanding the context

As a technical leader, you understand that many code issues follow patterns that can be automatically detected and evaluated, while other changes require human expertise. This automation tool needs to be able to analyze code changes, identify potential issues, and independently decide whether to approve changes, suggest fixes, or escalate to a human reviewer.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<div style="text-align: center;">
<img src="./images/agents-tasks-diagram.png" width=600>
</div>


<a id='2'></a>

## 2 - Set up your notebook

Begin by importing all necessary modules and configuring your environment variables to connect to the LLM APIs. 

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:

```Python
!pip install crewai[tools]==1.3.0
```

<a id='2-1'></a>

### 2.1 - Import modules

Run the following cell to import all the modules you will need for this lab. 

In [3]:
from crewai import Agent, Task, Crew
import dill
import unittests

Next, set up the environment variables to connect to the APIs, and  create the LLM instance you will use for your Agents

In [4]:
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set up the OpenAI model to use
os.environ["MODEL"] = 'gpt-4o-mini'

Before you start building your multi-agent model, review the pull request (PR) with the code changes from the `code_changes.txt` by running the cell below.

In [5]:
# read and save the policies content
with open('code_changes.txt', 'r') as file:
    code_changes = file.read()
print(code_changes)

diff --git a/app/user_auth.py b/app/user_auth.py
index 8f23c4d..b9e7f2a 100644
--- a/app/user_auth.py
+++ b/app/user_auth.py
@@ -1,7 +1,32 @@
+from datetime import datetime
+import time
+
 def authenticate_user(username, password):
+    # Check if username or password is empty
+    if not username or not password:
+        return False
+    
+    # Query the database for the user
     user = db.query(f"SELECT * FROM users WHERE username = '{username}'")
+    
+    # Verify the user exists and password matches
     if user and user.password == password:
+        # Set session variables
         session['user_id'] = user.id
+        session['login_time'] = datetime.now()
+        
+        # Update last login timestamp
+        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")
+        
+        print(f"User {username} logged in successfully")
         return True
-    return False
+    else:
+        # Sleep to prevent timing attacks
+        time.sleep(1)
+       

Now that you've broken down the problem and have a clearer understanding of its components, it's time to start creating the agents and tasks that will form your Crew.

<a id='3'></a>

## 3 - Define the elements for your Crew
You will need to define three agents for this Crew: 

- **Senior Developer**: A technical expert who examines code for style issues, potential bugs, and maintainability concerns, deciding which issues need to be addressed before approval. 
     
- **Security Engineer**: A security specialist who evaluates code changes for potential vulnerabilities, determining the risk level and whether security issues block approval. 
     
- **Tech Lead**: A decision-making leader who evaluates the findings from other agents, determines if a pull request can be automatically approved or needs human review, and provides final recommendations. 

<a id="3-1"></a>

### 3.1 Providing tools to your agents
In order to improve the performance of your agents, you can provide them with tools that allow them to connect to the external world. 

In order to follow best practices for security, you want to grant the **Security Engineer Agent** access to the **[OWASP](https://owasp.org)** webpage, a nonprofit foundation that works to improve the security of software.

<a id="ex1"></a>

### Exercise 1: Create tool instances

Create an instance of each of the following tools:
1. [**`SerperDevTool`**](https://docs.crewai.com/en/tools/search-research/serperdevtool). Use this tools to search within the OWASP webpage and retrieve the most relevant URLs for your problem. You will need to set the `search_url` to the OWASP URL. You can learn more about this tool in the **[docs](https://docs.crewai.com/en/tools/search-research/serperdevtool)**.

2. [**`ScrapeWebsiteTool`**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool). Use this tool to retrieve all the information from each of the identified websites. For more information, please refer to the **[docs](https://docs.crewai.com/en/tools/search-research/websitesearchtool)**.

`SerperDevTool` works similarly to the `ExaSearchTool`. It is designed to perform a semantic search for a specified query from a text’s content across the internet. It uses the serper.dev API to fetch and display the most relevant search results based on the query provided by the user. You will gain experience with this popular tool during this lab!

In [6]:
# GRADED CELL: Exercise 1
# import the required tools
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
# get the Serper API key
from utils import get_serper_api_key
serper_api_key = get_serper_api_key()

### START CODE HERE ###
# create the instance of the SerperDevTool. Set the search_url to "https://owasp.org"
serper_search_tool = SerperDevTool(search_url="https://owasp.org", base_url=os.getenv("DLAI_SERPER_BASE_URL"))

# create the instance of the ScrapeWebsiteTool, which does not need any arguments
scrape_website_tool = ScrapeWebsiteTool()
### END CODE HERE ###

In [7]:
# test the tools
unittests.test_tools(serper_search_tool, scrape_website_tool)

Using Tool: Search the internet with Serper
Using Tool: Read website content
 All tests passed!



<a id="3-2"></a>

### 3.2 Define the agents

Now it is time to define each agent. For each agent, you will need to specify four arguments:
- `role`: Their job title or function
- `goal`: What they aim to achieve
- `backstory`: Their experience and expertise (helps the LLM understand how to roleplay the agent)
- `verbose`: Whether to show detailed output (useful for learning and debugging)

Additionally, for the **Security Engineer** agent, you will need to assign the tools using the `tools` argument. 

<a id='ex2'></a>

### Exercise 2: Senior Developer agent

In the next cell, complete the `None` placeholders to create the **Senior Developer agent**.

Create an agent specialized in code quality evaluation by:

- Setting a `role` that reflects expertise in analyzing code quality.
- Defining a `goal` focused on evaluating code changes and deciding which issues must be fixed.
- Writing a `backstory` that emphasizes decision-making about code quality issues.

Make sure the agent understands it should determine which problems are critical vs. minor.

In [8]:
# GRADED CELL: Exercise 2

### START CODE HERE ### 

# Create the senior developer agent
senior_developer = Agent(
    role="Senior Developer",
    goal=(
        "Evaluate code changes for style issues, potential bugs, and maintainability concerns, "
        "and decide which issues are critical and must be fixed before approval and which are only minor suggestions."
    ),
    backstory=(
        "You are a senior developer with more than 15 years of experience reviewing code in large, fast-moving teams. "
        "You have a sharp eye for bugs, style violations, and maintainability problems, and you are decisive: "
        "you separate critical issues that must be fixed before a change can be approved from minor issues that are "
        "only suggested improvements, and you always explain your reasoning."
    ),
            
    # set verbose (suggested: True)
    verbose=True,
)

### END CODE HERE ###

In [9]:
# test the senior developer agent
unittests.test_senior_developer_agent(senior_developer)

 All tests passed!



<a id='ex3'></a>

### Exercise 3: Security Engineer agent
In the next cell, complete the `None` placeholders to create the **Security Engineer agent**.

Create an agent specialized in security analysis by:

- Setting a `role` that reflects expertise in code security evaluation.
- Defining a `goal` focused on identifying vulnerabilities and determining risk levels.
- Writing a `backstory` that emphasizes decision-making about security issues.
- Assigning the tools you created in [Exercise 1](#ex1)

Make sure the agent understands it should judge the severity of security concerns. 

In [10]:
# GRADED CELL: Exercise 3

### START CODE HERE ###

# Create the security engineer agent
security_engineer = Agent(
    role="Security Engineer",
    goal=(
        "Evaluate code changes for security vulnerabilities, determine the risk level of each issue, "
        "and decide whether the security issues should block approval of the change."
    ),
    backstory=( 
        "You are a security engineer with deep experience in application security and secure code review. "
        "You follow OWASP best practices and use them to judge how severe each vulnerability really is. "
        "You are decisive about which security concerns are serious enough to block a change and which are "
        "lower risk, and you always back your judgment with clear reasoning."
    ),
            
    # set verbose (suggested: True)
    verbose=True,
    # assign the tools you created in Exercise 1
    tools=[serper_search_tool, scrape_website_tool],
)

### END CODE HERE ###

In [11]:
# test the security engineer agent
unittests.test_security_engineer_agent(security_engineer)

 All tests passed!



<a id='ex4'></a>

### Exercise 4: Tech Lead agent

In the next cell, complete the `None` placeholders to create the **Tech Lead agent**.

Create an agent specialized in review coordination by:

- Setting a `role` that reflects expertise in managing code review processes.
- Defining a `goal` focused on determining approval paths for code changes.
- Writing a `backstory` that emphasizes decision-making about review workflows.

Make sure the agent understands it should make final judgments about approval or escalation.

In [13]:
# GRADED CELL: Exercise 4

### START CODE HERE ###

# Create the tech lead agent
tech_lead = Agent(
    role="Tech Lead",
    goal=(
        "Evaluate the code quality and security findings, decide whether a pull request can be automatically approved, "
        "needs specific fixes, or must be escalated to a human reviewer, and provide clear final recommendations."
    ),
    backstory=(
            "You are a tech lead who has managed code review processes for many years. You weigh the findings of the "
            "senior developer and the security engineer, balance delivery speed against code quality and security, "
            "and make the final call: approve automatically, request specific changes, or escalate to a human reviewer "
            "when the risk or uncertainty is too high. You always explain your decision."
        ),
        
    # set verbose (suggested: True)
    verbose=True,
)
### END CODE HERE ###

In [14]:
# test the tech lead agent
unittests.test_tech_lead_agent(tech_lead)

 All tests passed!



<a id='3-3'></a>

### 3.3 Define the Tasks for each Agent

Now that you have set up your agents, you are ready to define the tasks each of them will perform. In particular you will need three tasks (one for each agent):

- **Quality Analysis Task**: Evaluate code changes for style, bugs, and maintainability, deciding which issues must be fixed before approval. 
     
- **Security Review Task**: Examine code for security vulnerabilities, determining risk levels and whether security issues should block approval. 
     
- **Review Decision Task**: Analyze the quality and security findings to decide if changes can be automatically approved, need specific fixes, or require human review. 

#### General guidelines for creating Tasks:
When creating each task, you'll need to define these key parameters:

- `description`: A clear explanation of what the task involves
- `expected_output`: The format and content the task should produce
- `agent`: Which agent will perform this task
- `context` (optional): Define what tasks' output, including multiple, should be used as context for another task. You can learn more about context in the [docs](https://docs.crewai.com/en/concepts/tasks#referring-to-other-tasks). In this case, you will only need to set the context for the last task.

<a id='ex5'></a>

### Exercise 5: Create Quality Analysis Task

In the next cell, complete the `None` placeholders to create the **Quality Analysis task**.

Create a task for code quality evaluation by:
- Writing a `description` with steps instructing the agent to review code, identify potential bugs or issues, and decide if the issues are critical or minor.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a `JSON` with exactly these keys:
    - `critical_issues`: array of issues that must be fixed
    - `minor_issues`: array of suggested improvements
    - `reasoning`: explanation of decisions
- Assigning the task to the **Senior Developer** agent.

In [15]:
# GRADED CELL: Exercise 5

### START CODE HERE ###

# Create the quality analysis task
analyze_code_quality = Task(
    description=( 
        "Review the following code changes:\n\n{code_changes}\n\n"
        "Follow these steps:\n"
        "1. Carefully read through the code changes.\n"
        "2. Review the code for style issues, potential bugs, and maintainability concerns.\n"
        "3. For each issue you find, decide whether it is critical (must be fixed before approval) "
        "or minor (a suggested improvement).\n"
        "4. Explain the reasoning behind your decisions."
    ),
    
    expected_output=(
        "A JSON object with exactly these keys: "
        "critical_issues (an array of issues that must be fixed before approval), "
        "minor_issues (an array of suggested improvements), and "
        "reasoning (an explanation of the decisions). "
        "Return only the JSON object, with no extra text or code fences."
    ),
    
    name="Analyze Code Quality", #DO NOT CHANGE THIS NAME
    agent=senior_developer
)

### END CODE HERE ###

In [16]:
# test the quality analysis task
unittests.test_analyze_code_quality_task(analyze_code_quality)

 All tests passed!



<a id='ex6'></a>

### Exercise 6: Create Security Review Task

In the next cell, complete the `None` placeholders to create the **Security Review task**.

Create a task for security evaluation by:
- Completing the `description` with steps instructing the agent to examine code for vulnerabilities, identify security issues, determine risk levels, and decide if issues should block approval.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a `JSON` with exactly these keys:
    - `security_vulnerabilities`: array of identified issues with risk levels
    - `blocking`: boolean indicating if security issues should block approval
    - `highest_risk`: the most severe risk level found
    - `security_recommendations`: specific fixes for vulnerabilities
- Assigning the task to the **Security Engineer** agent.

In [17]:
# GRADED CELL: Exercise 6

### START CODE HERE ###

# Create the security review task
review_security = Task(
    description=( 
        "Review the following code changes for security issues:\n\n{code_changes}\n\n"
        "Follow these steps:\n"
        "1. Carefully read through the code changes and identify any security vulnerabilities.\n"
        "2. Determine the risk level of each vulnerability (low, medium, high, or critical).\n"
        "3. Decide whether any of the security issues should block approval of the change.\n"
        "4. Recommend specific fixes for each vulnerability.\n\n"
        # Add instructions for tool use to the task
        "Use the SerperDevTool to find the most relevant security best practices from OWASP"
        "and pass the URLs to the ScrapeWebsiteTool to get detailed information."
    ),
    
    expected_output=( 
        "A JSON object with exactly these keys: "
        "security_vulnerabilities (an array of the identified security issues, each with its risk level), "
        "blocking (a boolean indicating whether the security issues should block approval), "
        "highest_risk (the most severe risk level found), and "
        "security_recommendations (specific fixes for the vulnerabilities). "
        "Return only the JSON object, with no extra text or code fences."
    ),
    
    agent=security_engineer,
    name="Review Security", # DO NOT CHANGE THIS NAME
)

### END CODE HERE ###

In [18]:
# test the review security task
unittests.test_review_security_task(review_security)

 All tests passed!



<a id='ex7'></a>

### Exercise 7: Create Review Decision Task

In the next cell, complete the `None` placeholders to create the **Review Decision task**.

Create a task for review coordination by:
- Writing a `description` with steps instructing the agent to determine if the PR can be approved, decide on next steps, explain the decision.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a short report that includes the final decision, required changes (if any), approval comments (if approving), escalation reasoning (if escalating), and additional recommendations.
- Assigning the task to the **Tech Lead** agent.
- Assigning `context` to the agent. The Review Decision task needs the output of both of the previous tasks to make the decision, so you need to pass both tasks as context.

In [19]:
# GRADED CELL: Exercise 7

### START CODE HERE ###

# Create the review decision task
make_review_decision = Task(
    description=( 
        "Review the code quality and security findings provided as context for the following code changes:\n\n"
        "{code_changes}\n\n"
        "Follow these steps:\n"
        "1. Review the code quality findings, including the critical and minor issues.\n"
        "2. Review the security findings, including the vulnerabilities, their risk levels, and whether they are blocking.\n"
        "3. Decide whether the pull request can be automatically approved, needs specific fixes, "
        "or must be escalated to a human reviewer.\n"
        "4. Explain the reasoning behind your decision."
    ),
   
    expected_output=(
        "A short report that includes: the final decision (approve, request changes, or escalate to a human reviewer), "
        "the required changes (if any), approval comments (if approving), escalation reasoning (if escalating), "
        "and any additional recommendations."
    ), 
    
    agent=tech_lead,
    # add the two previous tasks as context
    context=[analyze_code_quality, review_security], 
    name="Review Decision", #DO NOT CHANGE THIS NAME
)

### END CODE HERE ###

In [20]:
# test the review decision task
unittests.test_make_review_decision_task(make_review_decision)

 All tests passed!



<a id='4'></a>

## 4 - Define and kick off your Crew

<a id='ex8'></a>

### Exercise 8: Define your Crew
Now that you have set up both agents and tasks, you are ready to put it all together and create your Crew! You will need to pass the `agents`, `tasks`, and `llm` you wish to use.

In [21]:
# GRADED CELL: Exercise 8

### START CODE HERE ###

# Create the code review crew
crew = Crew(
    # add the list of agents
    agents=[senior_developer, security_engineer, tech_lead],
    # add the list of tasks    
    tasks=[analyze_code_quality, review_security, make_review_decision],
)

### END CODE HERE ###

In [22]:
# test the crew
unittests.test_crew(crew)

 All tests passed!



Next, define all the inputs to kickoff your crew. Run the cell below to define the `inputs` dictionary with the `code_changes`, which you will then pass as context to your Crew.

In [23]:
# define the inputs dictionary for the crew
inputs = {
    "code_changes": code_changes,
}

<a id='ex9'></a>

### Exercise 9: Kickoff your Crew
Now you are ready to actually kickoff your crew and see it in action!

In [25]:
# GRADED CELL: Exercise 9

### START CODE HERE ###

# kickoff the crew
result = crew.kickoff(inputs=inputs)

### END CODE HERE ###

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Task: Review the following code changes:                                                                       │
│                                                                                                                 │
│  diff --git a/app/user_auth.py b/app/user_auth.py                                                               │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                            

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "critical_issues": [                                                                                         │
│      {                                                                                                          │
│        "issue": "SQL Injection vulnerability in user query.",                                                   │
│        "reasoning": "The code directly interpolates user input into the SQL query, making it susceptible to     │
│  SQL injection attacks. This must be fixed by using parameterized queries or ORM methods to avoid security      │
│  vulnerabilities."                                                                                              │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Unprotected access to user password.",                                                         │
│        "reasoning": "The comparison of the plaintext password against the stored password should not be done    │
│  without hashing the stored password. This poses a security risk and must be addressed before approval."        │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "minor_issues": [                                                                                            │
│      {                                                                                                          │
│        "issue": "Using print for logging.",                                                                     │
│        "reasoning": "Using print statements for logging is not a good practice. It's better to use a logging    │
│  library for better control over log levels and outputs."                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Sleep time for timing attacks.",                                                               │
│        "reasoning": "While the sleep function is intended to mitigate timing attacks, a better approach would   │
│  be just to return a generic failure response without an artificial delay. The current method could lead to     │
│  performance issues."                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Function documentation.",                                                                      │
│        "reasoning": "Adding docstrings or comments to the functions will improve maintainability and help       │
│  other developers understand the purpose and usage of t

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Task: Review the following code changes for security issues:                                                   │
│                                                                                                                 │
│  diff --git a/app/user_auth.py b/app/user_auth.py                                                               │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                            

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Thought: I need to gather more information regarding the identified security issues from OWASP to support my   │
│  decisions about their risk levels and necessary recommendations.                                               │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "OWASP best practices SQL Injection prevention"                                              │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {'searchParameters': {'q': 'OWASP best practices SQL Injection prevention', 'type': 'search', 'num': 10,       │
│  'engine': 'google'}, 'organic': [{'title': 'SQL Injection Prevention Cheat Sheet', 'link':                     │
│  'https://cheatsheetseries.owasp.org/cheatsheets/SQL_Injection_Prevention_Cheat_Sheet.html', 'snippet': 'To     │
│  minimize the potential damage of a successful SQL injection attack, you should minimize the privileges         │
│  assigned to every database account in your environment ...', 'position': 1}, {'title': 'SQL Injection | OWASP  │
│  Foundation', 'link': 'https://community.owasp.org/attacks/SQL_Injection', 'snippet': 'One traditional          │
│  approach to preventing SQL injection attacks is to handle them as an input validation problem and either       │
│  accept only characters from an allow ...', 'position': 2}, {'title': 'OWASP Top 10: Injection – What It Is     │
│  And How To Protect ...', 'link':                                                                               │
│  'https://cyolo.io/blog/owasp-top-10-injection-what-it-is-and-how-to-protect-our-applications', 'snippet':      │
│  "Now let's look at some best practices for how to prevent injection OWASP vulnerabilities: 1. Authorize        │
│  Users. Injection attacks are often ...", 'position': 3}, {'title': 'Injection Prevention Cheat Sheet',         │
│  'link': 'https://cheatsheetseries.owasp.org/cheatsheets/Injection_Prevention_Cheat_Sheet.html', 'snippet':     │
│  'This article is focused on providing clear, simple, actionable guidance for preventing injection flaws in     │
│  your applications. Injection attacks, for example SQL ...', 'position': 4}, {'title': 'Top 5 Best Practices    │
│  for Preventing SQL Injection Attacks', 'link':                                                                 │
│  'https://www.kiuwan.com/blog/top-5-best-practices-for-developers-on-preventing-sql-injections-attacks/',       │
│  'snippet': 'Learn how to prevent SQL injection with parameterized queries, input validation, least privilege,  │
│  secure configuration, and SAST tools like ...', 'position': 5}, {'title': 'SQL Injection attack prevention:    │
│  where do I start', 'link':                                                                                     │
│  'https://stackoverflow.com/questions/9162862/sql-injection-attack-prevention-where-do-i-start', 'snipp...      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (2 outputs left out)

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The following text is scraped website content:                                                                 │
│  SQL Injection Prevention - OWASP Cheat Sheet Series                                                            │
│  Skip to content                                                                                                │
│  OWASP Cheat Sheet Series                                                                                       │
│  SQL Injection Prevention                                                                                       │
│  Initializing search                                                                                            │
│  OWASP/CheatSheetSeries                                                                                         │
│  OWASP Cheat Sheet Series                                                                                       │
│  OWASP/CheatSheetSeries                                                                                         │
│  Introduction                                                                                                   │
│  Index Alphabetical                                                                                             │
│  Index ASVS                                                                                                     │
│  Index MASVS                                                                                                    │
│  Index Proactive Controls                                                                                       │
│  Index Top 10                                                                                                   │
│  Cheatsheets                                                                                                    │
│  Cheatsheets                                                                                                    │
│  AI-Powered Advertising Systems Security                                                                        │
│  AI Agent Security                                                                                              │
│  AML Sanctions AI Agent Payments                                                                                │
│  Abuse Case                                                                                                     │
│  Access Control                                                                                                 │
│  Attack Surface Analysis                                                                                        │
│  Authentication                                                                                                 │
│  Authentication Patterns                                                                                        │
│  Authorization                                                                                                  │
│  Authorization Decisions And Output Handling                                                                    │
│  Authorization Patterns                                                                                         │
│  Authorization Policy And Data Distribution                                                                     │
│  Authorization Regression Testing                                                                               │
│  Authorization Testing Automation                                                                               │
│  Automotive Security                                                                                            │
│  Bean Validation                                      

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the assessment of the change that was made in the code, the identified security issues, and the best  │
│  practices gathered, here is the summary of security vulnerabilities, their risk levels, whether they should    │
│  block approval, and the recommended fixes.                                                                     │
│                                                                                                                 │
│  {                                                                                                              │
│    "security_vulnerabilities": [                                                                                │
│      {                                                                                                          │
│        "issue": "SQL Injection vulnerability in user query.",                                                   │
│        "risk_level": "critical"                                                                                 │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Unprotected access to user password.",                                                         │
│        "risk_level": "critical"                                                                                 │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Using print for logging.",                                                                     │
│        "risk_level": "low"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Sleep time for timing attacks.",                                                               │
│        "risk_level": "low"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Function documentation.",                                                                      │
│        "risk_level": "low"                                                                                      │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "blocking": true,                                                                                            │
│    "highest_risk": "critical",                                                                                  │
│    "security_recommendations": [                       

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Task: Review the code quality and security findings provided as context for the following code changes:        │
│                                                                                                                 │
│  diff --git a/app/user_auth.py b/app/user_auth.py                                                               │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                            

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Final Decision:** Request changes                                                                            │
│                                                                                                                 │
│  **Required Changes:**                                                                                          │
│  1. **Fix SQL Injection Vulnerability:** The SQL query construction should use parameterized queries to         │
│  prevent SQL injection. Refactor the `db.query` and `db.execute` functions to properly bind parameters instead  │
│  of interpolating user inputs directly into the SQL string.                                                     │
│  2. **Secure Password Handling:** Passwords must be hashed before being stored or compared. Implement a secure  │
│  hashing algorithm (e.g., bcrypt) to hash user passwords, and update the authentication logic to compare        │
│  hashed values.                                                                                                 │
│  3. **Logging Improvement:** Replace `print` statements with a logging library, allowing better control over    │
│  log levels and outputs.                                                                                        │
│  4. **Timing Attack Mitigation:** Instead of using `time.sleep`, return a generic failure response to prevent   │
│  timing attacks without impacting performance.                                                                  │
│  5. **Function Documentation:** Add docstrings to the `authenticate_user` and `logout_user` functions to        │
│  clarify their purposes and usage.                                                                              │
│                                                                                                                 │
│  **Approval Comments:** The improvements suggested here will significantly enhance the security and             │
│  maintainability of the application. Addressing the critical vulnerabilities is mandatory before any approval,  │
│  as they pose severe risks to user data and system integrity.                                                   │
│                                                                                                                 │
│  **Escalation Reasoning:** Not applicable in this case, as the changes can be addressed by the current          │
│  development team.                                                                                              │
│                                                                                                                 │
│  **Additional Recommendations:**                                                                                │
│  - Implement unit tests to verify the security and functionality of the authentication process after the        │
│  recommended changes are complete.                                                                              │
│  - Consider conducting a security audit or pen-testing session after implementing the updates to ensure that    │
│  no vulnerabilities exist in the broader application context.                                                   │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

Let's check out the final report! Make sure all the information you requested is there. If not, you might need to rethink the task definition.

In [ ]:
from IPython.display import Markdown
Markdown(result.tasks_output[2].raw) 

Run the cell below to save the results, you will need this file for grading, so make sure to actually run the cell before you submit your work.

In [ ]:
with open("results.dill", "wb") as f:
    dill.dump(result, f)

Before submitting, you can check the output of the other two tasks have the desired format. Remember they were supposed to be dictionaries with specific keys.

**NOTE:** You will **NOT** be graded on whether your output is parseable JSON. LLMs don't always do this successfully. You'll learn more techniques for enforcing structured output in the coming modules!

Run the next cell to check the output of the first task (Analyze code Quality).

In [ ]:
from utils import get_dict_keys

# check the result of the first task

# Get the raw output
raw_output = result.tasks_output[0].raw

# See if it can be parsed as a dictionary, and get the keys
get_dict_keys(raw_output)

#### **Expected output:**

```
✅ Can be parsed as JSON dictionary
Keys: ['critical_issues', 'minor_issues', 'reasoning']
```

Now check the second task (Review Security).

In [ ]:
# check the result of the first task

# Get the raw output
raw_output = result.tasks_output[1].raw

# See if it can be parsed as a dictionary, and get the keys
get_dict_keys(raw_output)

#### **Expected output:**

```
✅ Can be parsed as JSON dictionary
Keys: ['security_vulnerabilities', 'blocking', 'highest_risk', 'security_recommendations']
```

You reached the end of the assignment. At this point you are ready to submit for grading. 

You can take some time to experiment with different definitions for your agents and tasks. You could also test it on your own commits and see how the answers change!